# MLP Nonlinearity Analysis

How the activation function transforms representations:
survival rates, distortion, magnitude shifts, and neuron selectivity.

## Why This Matters

MLP nonlinearity provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_nonlinearity_analysis import (
    activation_survival_rate, nonlinearity_distortion,
    activation_magnitude_shift, neuron_selectivity_profile,
    mlp_nonlinearity_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Activation Survival Rate

How many neurons survive the nonlinearity? Low survival = heavy filtering.

In [ ]:
for layer in range(4):
    result = activation_survival_rate(model, tokens, layer=layer)
    print(f"  Layer {layer}: survival={result['mean_survival_rate']:.4f}")
    for p in result['per_position']:
        bar = '█' * int(p['post_active_rate'] * 20)
        print(f"    Pos {p['position']}: pre+={p['pre_positive_rate']:.4f}, post+={p['post_active_rate']:.4f} {bar}")

## Nonlinearity Distortion

How much does the nonlinearity change the direction of activations?

In [ ]:
for layer in range(4):
    result = nonlinearity_distortion(model, tokens, layer=layer)
    flag = ' [LOW DIST]' if result['is_low_distortion'] else ''
    print(f"  Layer {layer}: cos={result['mean_cosine']:.4f}{flag}")
    for p in result['per_position']:
        print(f"    Pos {p['position']}: cos={p['cosine']:.4f}, norm_ratio={p['norm_ratio']:.4f}")

## Activation Magnitude Shift

How the nonlinearity scales magnitudes.

In [ ]:
for layer in range(4):
    result = activation_magnitude_shift(model, tokens, layer=layer)
    print(f"  Layer {layer}: pre_mean={result['pre_mean_magnitude']:.4f}, "
          f"post_mean={result['post_mean_magnitude']:.4f}, "
          f"ratio={result['magnitude_ratio']:.4f}")

## Neuron Selectivity

The most selective neurons — active for the fewest positions.

In [ ]:
for layer in range(4):
    result = neuron_selectivity_profile(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}: mean_selectivity={result['mean_selectivity']:.4f}, "
          f"always_active={result['n_always_active']}, never_active={result['n_never_active']}")
    for n in result['top_selective'][:3]:
        print(f"    Neuron {n['neuron']}: selectivity={n['selectivity']:.4f}, "
              f"active_positions={n['active_positions']}")

## Nonlinearity Summary

In [ ]:
result = mlp_nonlinearity_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [LOW DIST]' if p['is_low_distortion'] else ''
    print(f"  Layer {p['layer']}: survival={p['survival_rate']:.4f}, "
          f"distortion_cos={p['distortion_cosine']:.4f}{flag}")